# Metacentrum trace
 
Metacentrum cluster 17:

- Under investigation
- On the meanwhile, the Mustang specs will be used as a placeholder

Trace:

- 409949 Jobs sent to cluster 17 from 00:00:00 01 January 2023 (CET) to 23:55:15 31 December 2023 (CET)

In [ ]:
import json
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings



In [1]:
NUM_NODES = 47
NUM_CORES_PER_NODE = 64
CORE_FREQ = 2.6e9
CORE_FLOP_PER_CYCLE = 2

# The 3 parameters below have been copied from mustang.ipynb
IDLE_POWER_WATT = 10
EPSILON_POWER_WATT = 320
ALLCORES_POWER_WATT = 320

ZONE_ID = "CLUSTER-17"

In [ ]:
## metacentrum-cluster17.csv has been manually extracted from metacentrum2023.swf
df = pd.read_csv("datasets/metacentrum-cluster17.csv")
df.info()

### checking columns

In [ ]:
print(f"Loaded {len(df)} rows.")
print(f"Columns found: {list(df.columns)}")

# Check for expected columns
required_cols = ['job_id', 'submit_time', 'wait_time', 'runtime', 'allocated_CPU_cores', 'walltime_limit']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print(f"Warning: Missing columns: {missing}. Adjusting column names manually...")

# ---------------------------------------------------------------------------
# 2. NORMALIZE COLUMN NAMES
# ---------------------------------------------------------------------------
# Map your CSV headers to standard variable names used in the simulation logic
df.rename(columns={
    'allocated_CPU_cores': 'node_count',      # Use allocated cores as node count (assuming 1 core/node or uniform topology)
    'runtime': 'runtime_sec',                 # Runtime in seconds
    'walltime_limit': 'wallclock_limit',      # Wallclock limit in seconds
    'user_id': 'user_ID'                      # Ensure capitalization matches notebook later
}, inplace=True, errors='ignore')

# Ensure numeric types for time fields (crucial for math operations)
time_cols = ['submit_time', 'wait_time', 'runtime_sec', 'wallclock_limit']
for col in time_cols:
    if col in df.columns:
        # Convert to numeric, forcing non-numeric to NaN, then fill 0 if needed
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# ---------------------------------------------------------------------------
# 3. CALCULATE ABSOLUTE TIMES (DATETIME)
# ---------------------------------------------------------------------------
# Base timestamp from SWF header: UnixStartTime: 1672527600 (Jan 1, 2023 00:00:00 CET)
BASE_TIMESTAMP = 1672527600 

# Calculate start_time = submit_time + wait_time
# We do this by converting the SUM of seconds to a timedelta relative to BASE_TIMESTAMP
if 'submit_time' in df.columns and 'wait_time' in df.columns:
    # Calculate total offset in seconds first (scalar addition on series works fine)
    total_offset_seconds = df['submit_time'].astype(int) + df['wait_time'].astype(int)
    
    # Create datetime using vectorized conversion
    # Method: Convert base to datetime, then add array of timedeltas
    base_dt = pd.to_datetime(BASE_TIMESTAMP, unit='s')
    
    # Create a Series of timedeltas directly from the seconds
    offsets_td = pd.to_timedelta(total_offset_seconds, unit='s')
    df['start_time'] = base_dt + offsets_td
    
    # Calculate end_time = start_time + runtime
    runtime_td = pd.to_timedelta(df['runtime_sec'], unit='s')
    df['end_time'] = df['start_time'] + runtime_td
    
else:
    print("Error: 'submit_time' or 'wait_time' columns missing. Cannot calculate start/end times.")

# Ensure final time columns are datetime type
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])

# ---------------------------------------------------------------------------
# 4. DATA CLEANING
# ---------------------------------------------------------------------------
# Filter out invalid nodes (must be > 0 and <= configured max nodes later)
# For now, we just ensure node_count is positive
df = df[df['node_count'] > 0]

# Ensure runtime is positive
df = df[df['runtime_sec'] > 0]

# Check for negative wait times (should not exist in valid traces)
df = df[df['wait_time'] >= 0]

print("\nData processing complete.")
print(f"Date range: {df['start_time'].min()} to {df['end_time'].max()}")
print(f"Total jobs after cleaning: {len(df)}")
print("\nSample row:")
print(df[['job_id', 'submit_time', 'wait_time', 'start_time', 'end_time', 'runtime_sec', 'node_count']].head())

### Node speed calculation

In [ ]:
node_speed = NUM_CORES_PER_NODE * CORE_FREQ * CORE_FLOP_PER_CYCLE
print(f"Calculated Node Speed: {node_speed:.2e} flops/s")

### Ensuring data conversion is correct

In [ ]:
# Ensure timestamps are datetime
for col in ["submit_time", "start_time", "end_time"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True)

valid_nodes = (df["node_count"] > 0) & (df["node_count"] <= NUM_NODES)

# Filter running jobs
ran = df[valid_nodes & df["start_time"].notna() & (df["start_time"] <= df["end_time"])].copy()
ran["runtime_sec"] = (ran["end_time"] - ran["start_time"]).dt.total_seconds().clip(lower=1.0)

# Walltime logic
wallclock_sec = pd.to_timedelta(ran["wallclock_limit"], errors="coerce").dt.total_seconds()
# Handle cases where wallclock might be huge integers (seconds) instead of timedelta strings
# If wallclock_sec is NaN but wallclock_limit was an int, convert directly
if wallclock_sec.isna().all():
    wallclock_sec = ran["wallclock_limit"]
    
ran["walltime_sec"] = np.ceil(
    wallclock_sec.where(wallclock_sec > 0, ran["runtime_sec"]).clip(lower=ran["runtime_sec"])
)

# Filter queued jobs
queued = df[valid_nodes & df["submit_time"].notna()].copy()
queued["depart_time"] = queued["start_time"].fillna(queued["end_time"])
queued = queued[queued["submit_time"] <= queued["depart_time"]]

print(f"jobs kept for utilization: {len(ran)} / {len(df)}")
print(f"jobs kept for queue size:  {len(queued)} / {len(df)}")

### Visualization

In [ ]:
def node_series(arrive, depart, nodes, freq="1h"):
    """Nodes in use over time: +nodes at arrival, -nodes at departure,
    cumulative sum resampled to fixed bins."""
    events = pd.concat(
        [
            pd.Series(nodes.to_numpy(), index=arrive.to_numpy()),
            pd.Series(-nodes.to_numpy(), index=depart.to_numpy()),
        ]
    ).sort_index()
    return events.cumsum().resample(freq).last().ffill()


allocated = node_series(ran["start_time"], ran["end_time"], ran["node_count"])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(allocated.index, allocated.to_numpy(), color="black", linewidth=0.3)
ax.axhline(NUM_NODES, color="red", linewidth=1, label=f"Max Capacity ({NUM_NODES})")
ax.set_xlabel("Date")
ax.set_ylabel("Allocated Nodes")
ax.set_title("Utilization of MetaCentrum Cluster 17")
ax.legend()
ax.margins(x=0.01)
plt.show()

In [ ]:
in_queue = node_series(queued["submit_time"], queued["depart_time"], queued["node_count"])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(in_queue.index, in_queue.to_numpy(), color="gray", linewidth=0.3)
ax.set_xlabel("Date")
ax.set_ylabel("Nodes Requested in Queue")
ax.set_title("Queue size of MetaCentrum Cluster 17")
ax.margins(x=0.01)
plt.show()

## Selecting 4-week extracts for replay

Goal: pick 4-week windows to replay in Batsim for evaluating environmental-aware scheduling (carbon and water intensity signals) against FCFS and EASY backfilling baselines.

The selection criteria differ from the classic high-load replay recommendations. Since the heuristics work by shifting jobs in time or toggling backfilling, the windows must exercise both regimes. And since intensity signals vary on diurnal and weather timescales, a window must span multiple weeks, not hours.

Two windows are selected:

- **slack**: busy but not saturated. The friendly regime for time-shifting, the gentle one for backfill toggling.
- **stress**: heavily saturated. The regime where the FCFS vs EASY gap is widest and disabling backfilling has real consequences.

Search range: the whole trace, candidate windows anchored at Monday 00:00 UTC and slid by 1 week. The anchoring keeps the weekly phase identical across extracts and traces. The feasibility constraints below take care of excluding the pathological periods (ramp-up, drains, single-user bursts). Metrics per candidate window, all normalized by machine size so they transfer to other traces:

- `mean_util`, `std_util`: average and variability of node utilization.
- `frac_saturated`: fraction of hours at >= 95% capacity.
- `frac_low`: fraction of hours below 20% utilization.
- `max_low_streak_h`: longest contiguous run of low-utilization hours. Catches drain or maintenance events that a scattered-hours budget would let through.
- `mean_queue_norm`: average nodes requested in queue, in units of machine capacity.
- `jobs_submitted`, `top_user_share`: job-mix sanity. A window dominated by one user's burst makes results fragile.
- `wide_ns_share`: node-seconds share of wide jobs (>= 10% of the machine). Wide jobs create the reservations that make EASY differ from FCFS.
- `frac_bf_candidates`: share of narrow short jobs (<= 1% of the machine, <= 2 h), the jobs EASY can slip into reservation holes.

In [ ]:
WINDOW = pd.Timedelta(weeks=4)
WIDE_NODES = 0.10 * NUM_NODES
BF_NODES = 0.01 * NUM_NODES
BF_RUNTIME_SEC = 2 * 3600


def max_streak(mask):
    """Longest run of consecutive True values."""
    runs = mask.groupby((mask != mask.shift()).cumsum()).cumsum()
    return int(runs.max()) if len(runs) else 0


first_monday = (allocated.index[0].ceil("D") + pd.offsets.Week(weekday=0)).normalize()

rows = []
for start in pd.date_range(first_monday, allocated.index[-1] - WINDOW, freq="W-MON"):
    end = start + WINDOW
    util = allocated[start:end] / NUM_NODES
    replayed = ran[(ran["submit_time"] >= start) & (ran["submit_time"] < end)]
    node_sec = replayed["node_count"] * replayed["runtime_sec"]
    wide = replayed["node_count"] >= WIDE_NODES
    bf = (replayed["node_count"] <= BF_NODES) & (replayed["runtime_sec"] <= BF_RUNTIME_SEC)
    submitted = queued[(queued["submit_time"] >= start) & (queued["submit_time"] < end)]
    rows.append(
        {
            "start": start,
            "mean_util": util.mean(),
            "std_util": util.std(),
            "frac_saturated": (util >= 0.95).mean(),
            "frac_low": (util < 0.2).mean(),
            "max_low_streak_h": max_streak(util < 0.2),
            "mean_queue_norm": in_queue[start:end].mean() / NUM_NODES,
            "jobs_submitted": len(replayed),
            "top_user_share": submitted["user_ID"].value_counts(normalize=True).iloc[0]
            if len(submitted)
            else 1.0,
            "wide_ns_share": node_sec[wide].sum() / node_sec.sum() if node_sec.sum() else 0.0,
            "frac_bf_candidates": bf.mean() if len(replayed) else 0.0,
        }
    )
windows = pd.DataFrame(rows).set_index("start")
windows.round(3)

Feasibility constraints shared by both regimes:

- `frac_low <= 0.05` and `max_low_streak_h <= 6`: neither idle for long budgets nor hit by a contiguous drain.
- `top_user_share <= 0.5`: no single user dominates the job mix.
- `wide_ns_share >= 0.2` and `frac_bf_candidates >= 0.3`: enough wide jobs to force reservations and enough narrow short jobs to fill them, so EASY actually differs from FCFS.

Regime-specific constraints and ranking:

- **slack**: `0.5 <= mean_util <= 0.9` and `frac_saturated <= 0.3`. Ranked by `z(std_util) + z(mean_queue_norm)`: utilization variability indicates room to shift jobs, queue depth indicates sustained demand.
- **stress**: `mean_util <= 0.95` and `0.3 < frac_saturated <= 0.5`. Ranked by `z(frac_saturated) + z(mean_queue_norm)`: sustained pressure is the point of this regime.

In [ ]:
def zscore(s):
    return (s - s.mean()) / s.std() if s.std() > 0 else 0

common = (
    (windows["frac_low"] <= 0.10)              # Was 0.05, allow more idle time
    & (windows["max_low_streak_h"] <= 12)      # Was 6 hours, now 12
    & (windows["top_user_share"] <= 0.5)       # Keep same
    & (windows["wide_ns_share"] >= 0.1)        # Was 0.2, reduce for small cluster
    & (windows["frac_bf_candidates"] >= 0.2)   # Was 0.3, reduce slightly
    & (windows["jobs_submitted"] > 100)        # Must have meaningful job volume
)

print(f"Windows passing common constraints: {common.sum()}")

slack = windows[
    common 
    & windows["mean_util"].between(0.3, 0.85)  # Was 0.5-0.9
    & (windows["frac_saturated"] <= 0.4)       # More flexible saturation cap
].copy()

if len(slack) > 0:
    slack["score"] = zscore(slack["std_util"]) + zscore(slack["mean_queue_norm"])
    slack = slack.sort_values("score", ascending=False)
    print(f"Slack feasible windows: {len(slack)}")
else:
    print("No slack windows found with current constraints.")
    # If still empty, take best available window anyway
    slack = windows[common].sort_values("jobs_submitted", ascending=False).head(1)
    print(f"Using fallback: top {len(slack)} window(s) by job volume")

stress = windows[
    common
    & (windows["mean_util"] <= 0.90)           # Adjusted for realistic load
    & (windows["frac_saturated"] > 0.15)       # Was >0.3, hard to reach with 50 nodes
    & (windows["frac_saturated"] <= 0.6)       # Cap reasonable upper bound
].copy()

if len(stress) > 0:
    stress["score"] = zscore(stress["frac_saturated"]) + zscore(stress["mean_queue_norm"])
    stress = stress.sort_values("score", ascending=False)
    print(f"Stress feasible windows: {len(stress)}")
else:
    print("No stress windows found. Falling back to highest utilization window.")
    stress = windows[common].sort_values("mean_util", ascending=False).head(1)

# Ensure we have selections
EXTRACTS = {}
if len(slack) > 0:
    EXTRACTS["slack"] = (slack.index[0], slack.index[0] + WINDOW)
if len(stress) > 0:
    EXTRACTS["stress"] = (stress.index[0], stress.index[0] + WINDOW)

if not EXTRACTS:
    raise ValueError("No valid windows could be selected! Check 'windows' dataframe metrics above and consider further relaxing constraints.")

print(f"\nSelected extracts:")
for name, ranking in [("slack", slack), ("stress", stress)]:
    if name in EXTRACTS:
        t0, t1 = EXTRACTS[name]
        print(f"  {name}: {t0.date()} -> {t1.date()} ({len(ranking)} feasible windows)")

pd.concat([slack.head(3), stress.head(3)], keys=["slack", "stress"]).round(3)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)

axes[0].plot(windows.index, windows["mean_util"], marker=".", color="black")
axes[0].axhspan(0.5, 0.9, color="gray", alpha=0.2)
axes[0].set_ylabel("Mean Utilization")

axes[1].plot(windows.index, windows["frac_saturated"], marker=".", color="black", label="saturated (>= 95%)")
axes[1].plot(windows.index, windows["frac_low"], marker=".", color="gray", label="low (< 20%)")
axes[1].axhline(0.3, color="black", linewidth=0.8, linestyle="--")
axes[1].axhline(0.5, color="black", linewidth=0.8, linestyle=":")
axes[1].set_ylabel("Fraction of Hours")
axes[1].legend()

axes[2].plot(windows.index, windows["mean_queue_norm"], marker=".", color="black")
axes[2].set_ylabel("Mean Queue\n(machine units)")

axes[3].plot(windows.index, windows["wide_ns_share"], marker=".", color="black", label="wide node-sec share")
axes[3].plot(windows.index, windows["frac_bf_candidates"], marker=".", color="gray", label="backfill candidates")
axes[3].axhline(0.2, color="black", linewidth=0.8, linestyle="--")
axes[3].axhline(0.3, color="gray", linewidth=0.8, linestyle="--")
axes[3].set_ylabel("Job Mix")
axes[3].legend()

axes[4].plot(windows.index, windows["top_user_share"], marker=".", color="black")
axes[4].axhline(0.5, color="black", linewidth=0.8, linestyle="--")
axes[4].set_ylabel("Top User Share")
axes[4].set_xlabel("Window Start")

for ax in axes:
    ax.axvline(EXTRACTS["slack"][0], color="tab:red", linewidth=1)
    ax.axvline(EXTRACTS["stress"][0], color="tab:blue", linewidth=1)
fig.suptitle("Candidate 4-week windows (slack in red, stress in blue)")
plt.show()

In [ ]:
for name, (t0, t1) in EXTRACTS.items():
    fig, (ax_util, ax_queue) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    window_alloc = allocated[t0:t1]
    ax_util.plot(window_alloc.index, window_alloc.to_numpy(), color="black", linewidth=0.8)
    ax_util.axhline(NUM_NODES, color="black", linewidth=1)
    ax_util.set_ylabel("Allocated Nodes")

    window_queue = in_queue[t0:t1]
    ax_queue.plot(window_queue.index, window_queue.to_numpy(), color="black", linewidth=0.8)
    ax_queue.set_ylabel("Nodes Requested in Queue")
    ax_queue.set_xlabel("Date")

    fig.suptitle(f"{name} extract: {t0.date()} to {t1.date()}")
    plt.show()

## Batsim workload generation

One workload JSON per extract. Each job gets its own `parallel_homogeneous` profile with `cpu = runtime_sec * node_speed` and `com = 0` (network not modelled).

Jobs carry a walltime estimate because the EASY implementation (`easy_bf`) rejects jobs without one and builds its reservations from it. The estimate is the trace's `wallclock_limit` clipped so that `runtime <= walltime` always holds (walltime violations are not modelled, so TIMEOUT jobs that overran their limit in the real system get their walltime raised to the runtime).

> **Behavioral note**: because TIMEOUT jobs get `walltime = runtime` exactly, EASY's estimate for them is perfect, while every other job keeps its real, often loose, user estimate. This is the standard compromise when walltime violations are not modelled, but it slightly flatters backfilling accuracy for those jobs (about 7% of the trace).

Warm-up: the simulation starts with an empty machine, which the real system never was. To reproduce the system state at the extract start `T0`, all jobs that were in the system at `T0` are reset in the same state:

- **running at T0** (`start < T0 <= end`): submitted at `t = 0` with their *remaining* runtime and remaining walltime, ordered by original start time so FCFS reproduces the allocation order.
- **queued at T0** (`submit < T0 <= start`): submitted at `t = 0` with their full runtime, ordered by original submit time so they keep their queue positions. Jobs cancelled while queued are excluded since they have no defined runtime.

This defines two periods: the **warm-up** (environment setup at `t = 0`) and the **steady state**, which starts when the first replay job is submitted. Replay jobs are those submitted inside `[T0, T0 + 4 weeks)`, with `subtime` relative to `T0`. Evaluation metrics should be computed on the steady state only.

In [ ]:
def job_entry(job_id, subtime, res, runtime_sec, walltime_sec, jobs, profiles):
    profiles[job_id] = {
        "type": "parallel_homogeneous",
        "cpu": float(runtime_sec) * node_speed,
        "com": 0.0,
    }
    jobs.append(
        {
            "id": job_id,
            "subtime": float(subtime),
            "res": int(res),
            "walltime": int(max(walltime_sec, np.ceil(runtime_sec))),
            "profile": job_id,
        }
    )


def build_workload(t0, t1):
    """Batsim workload for the extract [t0, t1): warm-up context jobs at t=0,
    then the replayed jobs with their original arrival offsets."""
    running = ran[(ran["start_time"] < t0) & (ran["end_time"] >= t0)]
    waiting = ran[(ran["submit_time"] < t0) & (ran["start_time"] >= t0)]
    replay = ran[(ran["submit_time"] >= t0) & (ran["submit_time"] < t1)]

    jobs, profiles = [], {}
    for i, row in enumerate(running.sort_values("start_time").itertuples(), start=1):
        elapsed = (t0 - row.start_time).total_seconds()
        remaining = max((row.end_time - t0).total_seconds(), 1.0)
        job_entry(
            f"ctx_run{i}", 0.0, row.node_count, remaining,
            row.walltime_sec - elapsed, jobs, profiles,
        )
    for i, row in enumerate(waiting.sort_values("submit_time").itertuples(), start=1):
        job_entry(
            f"ctx_queued{i}", 0.0, row.node_count, row.runtime_sec,
            row.walltime_sec, jobs, profiles,
        )
    for i, row in enumerate(replay.sort_values("submit_time").itertuples(), start=1):
        subtime = (row.submit_time - t0).total_seconds()
        job_entry(
            f"job{i}", subtime, row.node_count, row.runtime_sec,
            row.walltime_sec, jobs, profiles,
        )

    return {"nb_res": NUM_NODES, "jobs": jobs, "profiles": profiles}


for name, (t0, t1) in EXTRACTS.items():
    workload = build_workload(t0, t1)
    out_path = f"metacentrum_{name}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(workload, f, indent=2)

    by_kind = {"ctx_run": 0, "ctx_queued": 0, "job": 0}
    for job in workload["jobs"]:
        by_kind[job["id"].rstrip("0123456789")] += 1
    nodes_running_t0 = sum(
        j["res"] for j in workload["jobs"] if j["id"].startswith("ctx_run")
    )
    steady_start = min(
        j["subtime"] for j in workload["jobs"] if j["id"].startswith("job")
    )
    print(
        f"{out_path}: {by_kind['ctx_run']} running + {by_kind['ctx_queued']} queued ctx jobs "
        f"({nodes_running_t0} nodes busy at t=0), {by_kind['job']} replay jobs, "
        f"steady state from t={steady_start:.0f}s"
    )

## SimGrid platform generation

Homogeneous platform modelled after Metacentrum. Nodes are modelled at full-node granularity: no `core` attribute, each host computes at `node_speed` flops. Power states follow the SimGrid `wattage_per_state` format `idle:epsilon:all_cores`.

In [ ]:
def create_prop(parent, prop_id, value):
    prop = ET.SubElement(parent, "prop")
    prop.set("id", prop_id)
    prop.set("value", value)


def generate_platform_xml(output_file, num_nodes=NUM_NODES):
    platform = ET.Element("platform")
    platform.set("version", "4.1")

    zone = ET.SubElement(platform, "zone")
    zone.set("id", ZONE_ID)
    zone.set("routing", "Full")

    # Management/login node, not a compute node. Batsim automatically assigns
    # the 'master' role to the master host.
    master = ET.SubElement(zone, "host")
    master.set("id", "master_host")
    create_prop(master, "role", "master")

    wattage = f"{IDLE_POWER_WATT}:{EPSILON_POWER_WATT}:{ALLCORES_POWER_WATT}"
    for i in range(num_nodes):
        node = ET.SubElement(zone, "host")
        node.set("id", f"node-{i}")
        node.set("speed", f"{node_speed / 1e9:g}Gf")
        create_prop(node, "role", "compute_node")
        create_prop(node, "wattage_per_state", wattage)

    tree = ET.ElementTree(platform)
    ET.indent(tree, space="    ")
    with open(output_file, "w") as f:
        f.write("<?xml version='1.0'?>\n")
        f.write('<!DOCTYPE platform SYSTEM "http://simgrid.gforge.inria.fr/simgrid/simgrid.dtd">\n')
        tree.write(f, encoding="unicode", xml_declaration=False)
    print(f"generated platform with {num_nodes} nodes at: {output_file}")


generate_platform_xml("metacentrum.xml")

# References

[1] G. Amvrosiadis, J. W. Park, G. R. Ganger, G. A. Gibson, E. Baseman, and N. DeBardeleben, “On the diversity of cluster workloads and its impact on research results,” in 2018 USENIX annual technical conference (USENIX ATC 18), Boston, MA: USENIX Association, Jul. 2018, pp. 533–546. [Online]. Available: https://www.usenix.org/conference/atc18/presentation/amvrosiadis

[2] J. Emeras, “Workload Traces Analysis and Replay in Large Scale Distributed Systems,” Université Grenoble Alpes, 2013.

[3] E. Frachtenberg and D. G. Feitelson, “Pitfalls in parallel job scheduling evaluation,” in Job scheduling strategies for parallel processing, vol. 3834, D. Feitelson, E. Frachtenberg, L. Rudolph, and U. Schwiegelshohn, Eds., in Lecture Notes in Computer Science, vol. 3834. , Berlin, Heidelberg: Springer Berlin Heidelberg, 2005, pp. 257–282. doi: 10.1007/11605300_13.

[4] Dalibor Klusáček and Václav Chlumský, "Real-life HPC Workload Trace Featuring Refined Job Runtime Estimates". In Job Scheduling Strategies for Parallel Processing, Springer, 2024. 